# 🩺 DiabetesPredict — Machine Learning Pipeline
**Diabetes Risk Prediction using the BRFSS 2021 dataset**

This notebook walks through the complete ML pipeline step by step:
data → preprocessing → training 3 models → evaluation → visualisation.

## Step 1 — Install & Import Libraries

In [ ]:
!pip install imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
import warnings; warnings.filterwarnings('ignore')
print('Libraries ready')

## Step 2 — Upload & Load the Dataset
Run the cell below and upload **diabetes_012_health_indicators_BRFSS2021.csv**.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('diabetes_012_health_indicators_BRFSS2021.csv')
df['Diabetes_012'] = df['Diabetes_012'].astype(int)
print('Dataset shape:', df.shape)
df.head()

## Step 3 — Exploratory Data Analysis (EDA)
Check for missing values and look at the class balance.

In [ ]:
labels = {0: 'No Diabetes', 1: 'Pre-Diabetes', 2: 'Diabetes'}
print('Missing values:', df.isnull().sum().sum())
print(df['Diabetes_012'].value_counts())

counts = df['Diabetes_012'].value_counts().sort_index()
plt.figure(figsize=(6, 4))
sns.barplot(x=[labels[i] for i in counts.index], y=counts.values,
            palette=['#22c55e', '#f59e0b', '#ef4444'])
plt.title('Class Distribution (imbalanced)'); plt.ylabel('Count'); plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap'); plt.show()

## Step 4 — Preprocessing (Split, SMOTE, Scaling)
- Split data into train/test
- **SMOTE** balances the classes
- **StandardScaler** normalises the features

In [ ]:
X = df.drop('Diabetes_012', axis=1)
y = df['Diabetes_012']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Before SMOTE:', X_train.shape)

X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)
print('After SMOTE :', X_train.shape)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Scaling done')

## Step 5 — Train & Compare 3 Models
Random Forest, Logistic Regression, and Decision Tree.

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=14,
                       min_samples_leaf=20, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=14,
                       min_samples_leaf=20, random_state=42),
}
results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_sc))
    results[name] = (model, acc)
    print(f'{name:22s}: {acc*100:.2f}%')

In [ ]:
print('Model comparison (best first):')
for name, (m, acc) in sorted(results.items(), key=lambda kv: kv[1][1], reverse=True):
    print(f'   {name:22s}: {acc*100:.2f}%')

best_name = max(results, key=lambda n: results[n][1])
best_model = results[best_name][0]
print('\nBest model:', best_name)

## Step 6 — Evaluate the Best Model
Classification report + confusion matrix.

In [ ]:
y_pred = best_model.predict(X_test_sc)
print(classification_report(y_test, y_pred, target_names=list(labels.values())))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels.values(), yticklabels=labels.values())
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()

## Step 7 — Cross-Validation (5-fold)
Confirms the model is stable (run on a 30k sample for speed).

In [ ]:
samp = df.sample(30000, random_state=1)
Xc = StandardScaler().fit_transform(samp.drop('Diabetes_012', axis=1))
yc = samp['Diabetes_012']
rf = RandomForestClassifier(n_estimators=80, max_depth=14,
                            min_samples_leaf=20, n_jobs=-1, random_state=42)
scores = cross_val_score(rf, Xc, yc, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print('Fold scores :', [f'{s*100:.1f}%' for s in scores])
print(f'Mean CV acc : {scores.mean()*100:.2f}%')

## Step 8 — Hyperparameter Tuning (GridSearchCV)

In [ ]:
grid = {'max_depth': [10, 14, 18], 'min_samples_leaf': [10, 20]}
gs = GridSearchCV(RandomForestClassifier(n_estimators=80, n_jobs=-1, random_state=42),
                  grid, cv=3)
gs.fit(Xc, yc)
print('Best parameters:', gs.best_params_)
print(f'Best CV accuracy: {gs.best_score_*100:.2f}%')

## Step 9 — Feature Importance

In [ ]:
imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(8, 7))
imp.plot(kind='barh', color='#6366f1')
plt.title('Feature Importance (Random Forest)'); plt.xlabel('Importance'); plt.show()

## Step 10 — PCA 2D Visualisation
Reduces 21 features to 2 dimensions to visualise the classes.

In [ ]:
samp2 = df.sample(3000, random_state=42)
Xs = StandardScaler().fit_transform(samp2.drop('Diabetes_012', axis=1))
pcs = PCA(n_components=2).fit_transform(Xs)
plt.figure(figsize=(7, 6))
for cls, c in zip([0, 1, 2], ['#22c55e', '#f59e0b', '#ef4444']):
    m = samp2['Diabetes_012'].values == cls
    plt.scatter(pcs[m, 0], pcs[m, 1], s=10, alpha=0.5, label=labels[cls], color=c)
plt.legend(); plt.title('PCA 2D Projection of Patients')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

## Step 11 — Save the Trained Model

In [ ]:
joblib.dump(best_model, 'diabetes_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Saved: diabetes_model.pkl + scaler.pkl')

## Step 12 — Test with an Example Patient

In [ ]:
example = pd.DataFrame([[1,1,1,40,1,0,1,0,0,0,0,1,0,5,15,20,1,1,11,3,2]],
                       columns=list(X.columns))
pred  = best_model.predict(scaler.transform(example))[0]
proba = best_model.predict_proba(scaler.transform(example))[0]
print('Prediction   :', labels[pred])
print('Probabilities:', dict(zip(labels.values(), (proba*100).round(1))))

## ✅ Conclusion
We trained and compared **3 models**, selected the most accurate (**Random Forest, ~80%**), validated it with cross-validation and tuning, and saved it as a `.pkl` file for use in the web application.